# Notebook 10 — Exploratory Machine Learning

## Metabolic transcriptomic signature classification in GSE136666

### Objective
Evaluate whether the compact, biologically informed metabolic signature prioritized across Notebooks 05–09 can discriminate Parkinson's disease (PD) from controls in substantia nigra.

### Critical interpretation
This notebook is **exploratory**, not a clinical classifier validation.

GSE136666 contains only **10 substantia nigra libraries (5 Control, 5 PD)**. Therefore:

- no 80/20 train/test split is used;
- no genome-wide feature selection is performed;
- only the pre-defined biologically informed signature is evaluated;
- leave-one-out cross-validation (LOOCV) is used;
- scaling is fitted only on training folds;
- feature subsets are defined by the mechanistic ranking from Notebook 09, not by outcome-driven selection inside this notebook;
- all tested feature-set/model combinations are reported;
- the highest CV score must not be interpreted as an unbiased estimate of future clinical performance.

Because the signature originated partly from GSE136666, classification performance on GSE136666 is **not independent validation**. The ML analysis asks whether the discovered signature has discriminative structure, not whether it is ready for diagnosis.

## 0. Imports, reproducibility, and project paths

In [ ]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    matthews_corrcoef,
)

RANDOM_STATE = 42

cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path(r"C:\Users\Camfe\pythonlab")

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

NB10_RESULTS = RESULTS_DIR / "machine_learning" / "notebook10"
NB10_FIGURES = FIGURES_DIR / "machine_learning" / "notebook10"

NB10_RESULTS.mkdir(parents=True, exist_ok=True)
NB10_FIGURES.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("NB10_RESULTS:", NB10_RESULTS)
print("NB10_FIGURES:", NB10_FIGURES)

## 1. Locate Notebook 09 feature candidates and GSE136666 inputs

In [ ]:
ML_FEATURE_FILE = (
    PROJECT_ROOT
    / "results"
    / "networks"
    / "notebook09"
    / "Notebook10_ML_feature_candidates.csv"
)

NB09_MANIFEST = (
    PROJECT_ROOT
    / "results"
    / "networks"
    / "notebook09"
    / "notebook09_manifest.json"
)

NB05_PRIORITIZED = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook05"
    / "GSE136666_SN_metabolic_candidates_prioritized.csv"
)

RAW_COUNTS_CANDIDATES = [
    PROJECT_ROOT / "data" / "GSE136666" / "GSE136666_raw_counts_GRCh38.p13_NCBI(1).tsv",
    PROJECT_ROOT / "data" / "GSE136666" / "GSE136666_raw_counts_GRCh38.p13_NCBI.tsv",
    PROJECT_ROOT / "GSE136666_raw_counts_GRCh38.p13_NCBI(1).tsv",
]

SERIES_CANDIDATES = [
    PROJECT_ROOT / "data" / "GSE136666" / "GSE136666_series_matrix(1).txt",
    PROJECT_ROOT / "data" / "GSE136666" / "GSE136666_series_matrix.txt",
    PROJECT_ROOT / "GSE136666_series_matrix(1).txt",
]

def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def fallback_find(path, pattern):
    path = Path(path)
    if path.exists():
        return path
    hits = list(PROJECT_ROOT.rglob(pattern))
    return hits[0] if hits else path

ML_FEATURE_FILE = fallback_find(
    ML_FEATURE_FILE,
    "Notebook10_ML_feature_candidates.csv",
)

NB09_MANIFEST = fallback_find(
    NB09_MANIFEST,
    "notebook09_manifest.json",
)

NB05_PRIORITIZED = fallback_find(
    NB05_PRIORITIZED,
    "GSE136666_SN_metabolic_candidates_prioritized.csv",
)

RAW_COUNTS_FILE = first_existing(RAW_COUNTS_CANDIDATES)

if RAW_COUNTS_FILE is None:
    hits = list(PROJECT_ROOT.rglob("GSE136666*raw*counts*.tsv"))
    RAW_COUNTS_FILE = hits[0] if hits else None

SERIES_FILE = first_existing(SERIES_CANDIDATES)

if SERIES_FILE is None:
    hits = list(PROJECT_ROOT.rglob("GSE136666*series*matrix*.txt"))
    SERIES_FILE = hits[0] if hits else None

print("ML_FEATURE_FILE:", ML_FEATURE_FILE, "|", ML_FEATURE_FILE.exists())
print("NB05_PRIORITIZED:", NB05_PRIORITIZED, "|", NB05_PRIORITIZED.exists())
print("RAW_COUNTS_FILE:", RAW_COUNTS_FILE)
print("SERIES_FILE:", SERIES_FILE)

## 2. Validate essential inputs

In [ ]:
required_paths = {
    "Notebook09 ML feature candidates": ML_FEATURE_FILE,
    "Notebook05 prioritized candidates": NB05_PRIORITIZED,
}

missing = [
    name
    for name, path in required_paths.items()
    if path is None or not Path(path).exists()
]

if RAW_COUNTS_FILE is None or not Path(RAW_COUNTS_FILE).exists():
    missing.append("GSE136666 raw counts")

if missing:
    raise FileNotFoundError(
        "Missing required Notebook 10 inputs:\n- "
        + "\n- ".join(missing)
    )

print("✓ Essential Notebook 10 inputs found.")

## 3. Load mechanistically prioritized feature candidates

In [ ]:
feature_candidates = pd.read_csv(
    ML_FEATURE_FILE
)

print("Shape:", feature_candidates.shape)
print("Columns:", feature_candidates.columns.tolist())

required = {
    "signature_gene",
    "mechanistic_priority_score",
    "mechanistic_rank",
}

missing = required.difference(
    feature_candidates.columns
)

if missing:
    raise KeyError(
        f"Notebook09 feature table is missing: {sorted(missing)}"
    )

feature_candidates["signature_gene"] = (
    feature_candidates["signature_gene"]
    .astype(str)
    .str.strip()
)

feature_candidates["mechanistic_rank"] = pd.to_numeric(
    feature_candidates["mechanistic_rank"],
    errors="coerce",
)

feature_candidates = (
    feature_candidates
    .sort_values("mechanistic_rank")
    .drop_duplicates("signature_gene")
    .reset_index(drop=True)
)

display(feature_candidates)

## 4. Load Notebook 05 mapping table

The bulk count matrix uses Entrez/GeneID. Notebook 05 provides the bridge between readable gene symbols and GeneID.

In [ ]:
prioritized = pd.read_csv(
    NB05_PRIORITIZED
)

print("Shape:", prioritized.shape)
print("Columns:")
print(prioritized.columns.tolist())

def detect_column(df, preferred_names, contains=None, required=False, label="column"):
    lower_map = {c.lower(): c for c in df.columns}

    for name in preferred_names:
        if name.lower() in lower_map:
            return lower_map[name.lower()]

    if contains:
        for c in df.columns:
            cl = c.lower()
            if all(token.lower() in cl for token in contains):
                return c

    if required:
        raise KeyError(
            f"Could not detect required {label}. "
            f"Available columns: {df.columns.tolist()}"
        )

    return None

SYMBOL_COL = detect_column(
    prioritized,
    ["Symbol", "symbol", "gene", "gene_symbol", "GeneSymbol"],
    contains=["symbol"],
    required=True,
    label="gene symbol",
)

GENEID_COL = detect_column(
    prioritized,
    ["GeneID", "gene_id", "Entrez", "entrez", "EntrezID"],
    contains=["gene", "id"],
    required=True,
    label="GeneID",
)

print("SYMBOL_COL:", SYMBOL_COL)
print("GENEID_COL:", GENEID_COL)

mapping = (
    prioritized[
        [SYMBOL_COL, GENEID_COL]
    ]
    .dropna()
    .copy()
)

mapping["signature_gene"] = (
    mapping[SYMBOL_COL]
    .astype(str)
    .str.strip()
)

mapping["GeneID_clean"] = (
    mapping[GENEID_COL]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

mapping = (
    mapping[
        ["signature_gene", "GeneID_clean"]
    ]
    .drop_duplicates()
)

feature_map = feature_candidates.merge(
    mapping,
    on="signature_gene",
    how="left",
)

display(feature_map)

if feature_map["GeneID_clean"].isna().any():
    print("\nWARNING: Some candidate genes lack GeneID mapping:")
    display(
        feature_map.loc[
            feature_map["GeneID_clean"].isna(),
            ["signature_gene", "mechanistic_rank"]
        ]
    )

## 5. Load raw GSE136666 counts

In [ ]:
raw_counts = pd.read_csv(
    RAW_COUNTS_FILE,
    sep="\t",
)

print("Raw shape:", raw_counts.shape)
print(raw_counts.head())

gene_id_candidates = [
    c for c in raw_counts.columns
    if c.lower() in {
        "geneid",
        "gene_id",
        "entrez",
        "entrezid",
    }
]

if gene_id_candidates:
    RAW_GENEID_COL = gene_id_candidates[0]
else:
    RAW_GENEID_COL = raw_counts.columns[0]

raw_counts["GeneID_clean"] = (
    raw_counts[RAW_GENEID_COL]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

print("Detected raw GeneID column:", RAW_GENEID_COL)
print("Unique GeneIDs:", raw_counts["GeneID_clean"].nunique())

## 6. Identify substantia nigra sample columns and labels

The known study design contains:
- `SN_C_1` ... `SN_C_5` = Control
- `SN_PD_1` ... `SN_PD_5` = Parkinson

In [ ]:
sn_control_cols = [
    c for c in raw_counts.columns
    if re.fullmatch(r"SN_C_[1-5]", str(c))
]

sn_pd_cols = [
    c for c in raw_counts.columns
    if re.fullmatch(r"SN_PD_[1-5]", str(c))
]

sn_control_cols = sorted(sn_control_cols)
sn_pd_cols = sorted(sn_pd_cols)

print("Control SN columns:", sn_control_cols)
print("PD SN columns:", sn_pd_cols)

if len(sn_control_cols) != 5 or len(sn_pd_cols) != 5:
    raise RuntimeError(
        "Expected 5 SN Control and 5 SN PD columns. "
        "Inspect raw count column names."
    )

sample_cols = sn_control_cols + sn_pd_cols

sample_metadata = pd.DataFrame({
    "sample": sample_cols,
    "disease": (
        ["Control"] * len(sn_control_cols)
        +
        ["PD"] * len(sn_pd_cols)
    ),
})

sample_metadata["y"] = (
    sample_metadata["disease"]
    .map({"Control": 0, "PD": 1})
    .astype(int)
)

display(sample_metadata)

## 7. Extract only the biologically pre-defined genes

No outcome-based feature selection is performed here.

In [ ]:
selected_map = (
    feature_map
    .dropna(subset=["GeneID_clean"])
    .copy()
)

selected_ids = set(
    selected_map["GeneID_clean"]
)

subset_counts = (
    raw_counts.loc[
        raw_counts["GeneID_clean"].isin(selected_ids),
        ["GeneID_clean"] + sample_cols,
    ]
    .copy()
)

print(
    "Mapped candidate genes found in raw counts:",
    subset_counts["GeneID_clean"].nunique(),
    "/",
    selected_map["GeneID_clean"].nunique(),
)

# If duplicate GeneIDs occur, sum them conservatively
subset_counts = (
    subset_counts
    .groupby("GeneID_clean", as_index=False)[sample_cols]
    .sum()
)

gene_to_id = (
    selected_map[
        ["signature_gene", "GeneID_clean", "mechanistic_rank"]
    ]
    .drop_duplicates("signature_gene")
)

# Long mapping check
mapping_check = gene_to_id.merge(
    subset_counts[["GeneID_clean"]],
    on="GeneID_clean",
    how="left",
    indicator=True,
)

display(mapping_check)

missing_expression = mapping_check.loc[
    mapping_check["_merge"] != "both"
]

if len(missing_expression):
    print("\nGenes missing from raw counts:")
    display(missing_expression)

if subset_counts["GeneID_clean"].nunique() < 3:
    raise RuntimeError(
        "Fewer than 3 candidate genes are present in the raw bulk matrix."
    )

## 8. Build samples × genes raw-count matrix

In [ ]:
expr_by_id = (
    subset_counts
    .set_index("GeneID_clean")[sample_cols]
    .T
)

id_to_gene = (
    gene_to_id
    .drop_duplicates("GeneID_clean")
    .set_index("GeneID_clean")["signature_gene"]
    .to_dict()
)

usable_ids = [
    c for c in expr_by_id.columns
    if c in id_to_gene
]

X_raw = expr_by_id[usable_ids].copy()

X_raw.columns = [
    id_to_gene[c]
    for c in X_raw.columns
]

# Order by mechanistic rank
rank_lookup = (
    feature_candidates
    .set_index("signature_gene")["mechanistic_rank"]
    .to_dict()
)

ordered_genes = sorted(
    X_raw.columns,
    key=lambda g: rank_lookup.get(g, np.inf),
)

X_raw = X_raw[ordered_genes]

X_raw = X_raw.loc[
    sample_metadata["sample"]
]

y = sample_metadata.set_index("sample").loc[
    X_raw.index,
    "y"
].to_numpy()

print("X_raw shape:", X_raw.shape)
print("Genes:", X_raw.columns.tolist())
print("Class counts:", pd.Series(y).value_counts().to_dict())

display(X_raw)

## 9. Sample-wise normalization for exploratory ML

For ML we use **log2 counts-per-million (log-CPM)**.

Why:
- raw library sizes differ;
- CPM normalization is calculated independently for each sample;
- it therefore does not use held-out labels or test-set distribution;
- StandardScaler is still fitted only inside each training fold.

This transformation is for exploratory classification only.  
It does not replace DESeq2 normalization used for differential-expression inference.

In [ ]:
library_sizes = X_raw.sum(axis=1)

if (library_sizes <= 0).any():
    raise ValueError("A sample has non-positive selected-gene library size.")

# IMPORTANT:
# True whole-library CPM should use total raw library size from all genes,
# not only the selected feature genes.
whole_library_sizes = (
    raw_counts[sample_cols]
    .sum(axis=0)
    .astype(float)
)

whole_library_sizes = (
    whole_library_sizes
    .reindex(X_raw.index)
)

if whole_library_sizes.isna().any():
    raise ValueError(
        "Could not align total library sizes to SN samples."
    )

cpm = (
    X_raw
    .div(whole_library_sizes, axis=0)
    * 1_000_000.0
)

X_logcpm = np.log2(
    cpm + 1.0
)

print("Whole-library sizes:")
display(whole_library_sizes.to_frame("total_counts"))

print("\nlog2(CPM+1) feature matrix:")
display(X_logcpm)

## 10. Define mechanistic feature sets

Feature subsets are pre-defined from Notebook 09 ranking:

- Top 3
- Top 5
- Top 10
- All available signature genes

These are **not chosen by their classification performance**.

In [ ]:
n_available = X_logcpm.shape[1]

requested_sizes = [
    3,
    5,
    10,
    n_available,
]

requested_sizes = sorted(
    set(
        min(n_available, n)
        for n in requested_sizes
        if min(n_available, n) >= 1
    )
)

feature_sets = {}

for n in requested_sizes:
    feature_sets[
        f"Top_{n}"
    ] = X_logcpm.columns[:n].tolist()

print("Feature sets:")

for name, genes in feature_sets.items():
    print(name, "->", genes)

## 11. Define intentionally simple classifiers

With n=10, model complexity must remain low.

Evaluated:
- Logistic Regression (L2)
- Linear SVM
- Ridge Classifier
- Random Forest as a nonlinear exploratory comparator

The linear models are the primary models. Random Forest is secondary because tree ensembles are unstable with extremely small n.

In [ ]:
models = {
    "LogisticRegression_L2": Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                C=1.0,
                solver="liblinear",
                class_weight="balanced",
                max_iter=5000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "Linear_SVM": Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "model",
            SVC(
                kernel="linear",
                C=1.0,
                class_weight="balanced",
                probability=False,
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "RidgeClassifier": Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "model",
            RidgeClassifier(
                alpha=1.0,
                class_weight="balanced",
            ),
        ),
    ]),

    "RandomForest_secondary": RandomForestClassifier(
        n_estimators=500,
        max_depth=2,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}

print("Models:")
for name in models:
    print("-", name)

## 12. LOOCV evaluation function

Every sample is held out exactly once.

For each fold:
1. training samples are selected;
2. scaler/model are fitted on training data only;
3. the held-out sample is predicted;
4. prediction and continuous score are stored.

No preprocessing parameter is fitted using all 10 samples.

In [ ]:
def get_continuous_score(model, X_test):
    """
    Continuous score for ROC-AUC.
    """
    if hasattr(model, "predict_proba"):
        return float(
            model.predict_proba(X_test)[0, 1]
        )

    if hasattr(model, "decision_function"):
        score = model.decision_function(X_test)

        if np.ndim(score) == 0:
            return float(score)

        return float(np.ravel(score)[0])

    return float(model.predict(X_test)[0])


def evaluate_loocv(X_df, y, estimator):
    loo = LeaveOneOut()

    rows = []

    X_arr = X_df.to_numpy(dtype=float)

    for fold, (train_idx, test_idx) in enumerate(
        loo.split(X_arr),
        start=1,
    ):

        model = clone(estimator)

        X_train = X_arr[train_idx]
        X_test = X_arr[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        model.fit(
            X_train,
            y_train,
        )

        pred = int(
            model.predict(X_test)[0]
        )

        score = get_continuous_score(
            model,
            X_test,
        )

        rows.append({
            "fold": fold,
            "test_index": int(test_idx[0]),
            "sample": X_df.index[test_idx[0]],
            "true": int(y_test[0]),
            "pred": pred,
            "score": score,
        })

    pred_df = pd.DataFrame(rows)

    y_true = pred_df["true"].to_numpy()
    y_pred = pred_df["pred"].to_numpy()
    y_score = pred_df["score"].to_numpy()

    metrics = {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "mcc": matthews_corrcoef(
            y_true,
            y_pred,
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_score,
        ),
    }

    return pred_df, metrics

## 13. Run all pre-defined feature-set × model combinations

In [ ]:
all_predictions = []
metric_rows = []

for feature_set_name, genes in feature_sets.items():

    X_sub = X_logcpm[genes]

    for model_name, estimator in models.items():

        print(
            f"Running {feature_set_name} | {model_name}"
        )

        pred_df, metrics = evaluate_loocv(
            X_sub,
            y,
            estimator,
        )

        pred_df["feature_set"] = (
            feature_set_name
        )

        pred_df["model"] = (
            model_name
        )

        all_predictions.append(
            pred_df
        )

        metric_rows.append({
            "feature_set": feature_set_name,
            "n_features": len(genes),
            "genes": "; ".join(genes),
            "model": model_name,
            **metrics,
        })

predictions = pd.concat(
    all_predictions,
    ignore_index=True,
)

metrics_table = pd.DataFrame(
    metric_rows
)

metrics_table = metrics_table.sort_values(
    [
        "roc_auc",
        "balanced_accuracy",
        "f1",
    ],
    ascending=False,
)

predictions.to_csv(
    NB10_RESULTS /
    "LOOCV_predictions_all_models.csv",
    index=False,
)

metrics_table.to_csv(
    NB10_RESULTS /
    "LOOCV_metrics_all_models.csv",
    index=False,
)

display(metrics_table)

## 14. Important anti-selection warning

The table above contains multiple model/feature-set comparisons.

The row with the numerically highest AUC is **not** an unbiased estimate of performance because:
- the same tiny dataset is used to compare configurations;
- the signature itself was partly discovered in GSE136666.

We therefore report all configurations and designate a primary analysis a priori:

**Top-5 mechanistic genes + Logistic Regression L2**

In [ ]:
PRIMARY_FEATURE_SET = (
    "Top_5"
    if "Top_5" in feature_sets
    else list(feature_sets.keys())[0]
)

PRIMARY_MODEL = (
    "LogisticRegression_L2"
)

primary_metrics = metrics_table.loc[
    (
        metrics_table["feature_set"]
        ==
        PRIMARY_FEATURE_SET
    )
    &
    (
        metrics_table["model"]
        ==
        PRIMARY_MODEL
    )
]

print("Primary ML analysis:")
print(PRIMARY_FEATURE_SET, "+", PRIMARY_MODEL)

display(primary_metrics)

## 15. Primary LOOCV predictions

In [ ]:
primary_predictions = predictions.loc[
    (
        predictions["feature_set"]
        ==
        PRIMARY_FEATURE_SET
    )
    &
    (
        predictions["model"]
        ==
        PRIMARY_MODEL
    )
].copy()

primary_predictions["true_label"] = (
    primary_predictions["true"]
    .map({
        0: "Control",
        1: "PD",
    })
)

primary_predictions["pred_label"] = (
    primary_predictions["pred"]
    .map({
        0: "Control",
        1: "PD",
    })
)

display(primary_predictions)

primary_predictions.to_csv(
    NB10_RESULTS /
    "primary_LOOCV_predictions.csv",
    index=False,
)

## 16. Primary confusion matrix

In [ ]:
cm = confusion_matrix(
    primary_predictions["true"],
    primary_predictions["pred"],
    labels=[0, 1],
)

fig, ax = plt.subplots(
    figsize=(4.5, 4)
)

im = ax.imshow(
    cm,
    aspect="equal",
)

ax.set_xticks([0, 1])
ax.set_xticklabels(
    ["Control", "PD"]
)

ax.set_yticks([0, 1])
ax.set_yticklabels(
    ["Control", "PD"]
)

for i in range(2):
    for j in range(2):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            fontsize=12,
        )

ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(
    "Primary LOOCV confusion matrix"
)

fig.tight_layout()

fig.savefig(
    NB10_FIGURES /
    "primary_LOOCV_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

## 17. Primary ROC curve

In [ ]:
fpr, tpr, thresholds = roc_curve(
    primary_predictions["true"],
    primary_predictions["score"],
)

primary_auc = roc_auc_score(
    primary_predictions["true"],
    primary_predictions["score"],
)

fig, ax = plt.subplots(
    figsize=(5, 5)
)

ax.plot(
    fpr,
    tpr,
    marker="o",
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
)

ax.set_xlabel(
    "False positive rate"
)

ax.set_ylabel(
    "True positive rate"
)

ax.set_title(
    f"Primary LOOCV ROC — AUC={primary_auc:.3f}"
)

fig.tight_layout()

fig.savefig(
    NB10_FIGURES /
    "primary_LOOCV_ROC.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

## 18. Compare models and feature-set sizes

This visualization is descriptive and should not be used to select a 'winning' classifier for clinical claims.

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 5)
)

plot_table = (
    metrics_table
    .copy()
)

x_labels = (
    plot_table["feature_set"]
    +
    "\n"
    +
    plot_table["model"]
)

ax.bar(
    np.arange(len(plot_table)),
    plot_table["roc_auc"],
)

ax.set_xticks(
    np.arange(len(plot_table))
)

ax.set_xticklabels(
    x_labels,
    rotation=90,
    fontsize=7,
)

ax.set_ylabel("LOOCV ROC-AUC")

ax.set_title(
    "Exploratory comparison of pre-defined ML configurations"
)

ax.set_ylim(0, 1.05)

fig.tight_layout()

fig.savefig(
    NB10_FIGURES /
    "LOOCV_AUC_model_comparison.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

## 19. Permutation test for the primary configuration

A label-permutation test asks whether the observed LOOCV AUC is unusually high relative to randomly permuted labels.

Because n=10 is tiny, this remains exploratory.

The feature set is kept fixed. Only labels are permuted.

In [ ]:
N_PERMUTATIONS = 1000

rng = np.random.default_rng(
    RANDOM_STATE
)

primary_genes = feature_sets[
    PRIMARY_FEATURE_SET
]

X_primary = X_logcpm[
    primary_genes
]

primary_estimator = models[
    PRIMARY_MODEL
]

observed_auc = float(
    primary_metrics.iloc[0]["roc_auc"]
)

permuted_aucs = []

for i in range(
    N_PERMUTATIONS
):

    y_perm = rng.permutation(
        y
    )

    _, perm_metrics = evaluate_loocv(
        X_primary,
        y_perm,
        primary_estimator,
    )

    permuted_aucs.append(
        perm_metrics["roc_auc"]
    )

permuted_aucs = np.array(
    permuted_aucs,
    dtype=float,
)

permutation_p = (
    1
    +
    np.sum(
        permuted_aucs
        >=
        observed_auc
    )
) / (
    N_PERMUTATIONS
    +
    1
)

print(
    "Observed LOOCV AUC:",
    observed_auc,
)

print(
    "Permutation p-value:",
    permutation_p,
)

permutation_results = pd.DataFrame({
    "permuted_auc": permuted_aucs,
})

permutation_results.to_csv(
    NB10_RESULTS /
    "primary_LOOCV_AUC_permutations.csv",
    index=False,
)

## 20. Permutation distribution

In [ ]:
fig, ax = plt.subplots(
    figsize=(6, 4.5)
)

ax.hist(
    permuted_aucs,
    bins=20,
)

ax.axvline(
    observed_auc,
    linewidth=2,
)

ax.set_xlabel(
    "Permuted-label LOOCV AUC"
)

ax.set_ylabel(
    "Frequency"
)

ax.set_title(
    f"Permutation test — p={permutation_p:.4f}"
)

fig.tight_layout()

fig.savefig(
    NB10_FIGURES /
    "primary_AUC_permutation_test.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

## 21. Exact/near-exact uncertainty by bootstrap over LOOCV predictions

The 10 LOOCV predictions are not a new independent cohort, so this bootstrap is shown only as an approximate descriptive uncertainty interval.

It must not be presented as an external-validation confidence interval.

In [ ]:
N_BOOTSTRAPS = 5000

rng = np.random.default_rng(
    RANDOM_STATE + 1
)

y_true_primary = (
    primary_predictions["true"]
    .to_numpy()
)

y_score_primary = (
    primary_predictions["score"]
    .to_numpy()
)

bootstrap_aucs = []

n = len(
    y_true_primary
)

for _ in range(
    N_BOOTSTRAPS
):

    idx = rng.integers(
        0,
        n,
        size=n,
    )

    y_b = y_true_primary[
        idx
    ]

    s_b = y_score_primary[
        idx
    ]

    if len(
        np.unique(
            y_b
        )
    ) < 2:
        continue

    bootstrap_aucs.append(
        roc_auc_score(
            y_b,
            s_b,
        )
    )

bootstrap_aucs = np.array(
    bootstrap_aucs
)

auc_ci_low = float(
    np.percentile(
        bootstrap_aucs,
        2.5,
    )
)

auc_ci_high = float(
    np.percentile(
        bootstrap_aucs,
        97.5,
    )
)

print(
    "Approximate bootstrap AUC interval:",
    f"[{auc_ci_low:.3f}, {auc_ci_high:.3f}]"
)

## 22. Fit the primary linear model on all 10 samples only for coefficient interpretation

This final fit is **not** used to estimate generalization performance.

It is used only to inspect the direction of standardized logistic-regression coefficients.

In [ ]:
primary_final_model = clone(
    primary_estimator
)

primary_final_model.fit(
    X_primary,
    y,
)

if isinstance(
    primary_final_model,
    Pipeline,
):

    logistic = (
        primary_final_model
        .named_steps["model"]
    )

    coefficients = (
        logistic.coef_[0]
    )

    coef_table = pd.DataFrame({
        "gene": primary_genes,
        "standardized_logistic_coefficient": coefficients,
    })

    coef_table[
        "abs_coefficient"
    ] = (
        coef_table[
            "standardized_logistic_coefficient"
        ]
        .abs()
    )

    coef_table = coef_table.sort_values(
        "abs_coefficient",
        ascending=False,
    )

    coef_table.to_csv(
        NB10_RESULTS /
        "primary_logistic_coefficients.csv",
        index=False,
    )

    display(coef_table)

## 23. Visualize primary logistic coefficients

Positive coefficients are associated with the PD class within this tiny exploratory dataset; negative coefficients with Control.

Coefficient direction must not be interpreted causally.

In [ ]:
if "coef_table" in globals():

    plot_coef = (
        coef_table
        .sort_values(
            "standardized_logistic_coefficient"
        )
    )

    fig, ax = plt.subplots(
        figsize=(6, 4.5)
    )

    ax.barh(
        plot_coef["gene"],
        plot_coef[
            "standardized_logistic_coefficient"
        ],
    )

    ax.axvline(
        0,
        linewidth=0.8,
    )

    ax.set_xlabel(
        "Standardized logistic coefficient"
    )

    ax.set_title(
        "Primary model coefficients"
    )

    fig.tight_layout()

    fig.savefig(
        NB10_FIGURES /
        "primary_logistic_coefficients.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

## 24. Sample-level feature heatmap

This is descriptive visualization of the selected primary genes after log-CPM transformation and per-gene z-scoring across samples.

This visualization scaling is separate from the cross-validation pipeline and is not used for model fitting.

In [ ]:
heatmap_df = X_primary.copy()

heatmap_z = (
    heatmap_df
    -
    heatmap_df.mean(axis=0)
) / (
    heatmap_df.std(
        axis=0,
        ddof=0,
    )
    .replace(0, 1)
)

fig, ax = plt.subplots(
    figsize=(
        max(6, len(primary_genes) * 0.9),
        6,
    )
)

im = ax.imshow(
    heatmap_z.to_numpy(),
    aspect="auto",
)

ax.set_xticks(
    np.arange(
        len(primary_genes)
    )
)

ax.set_xticklabels(
    primary_genes,
    rotation=45,
    ha="right",
)

ax.set_yticks(
    np.arange(
        len(heatmap_z)
    )
)

ax.set_yticklabels(
    heatmap_z.index
)

ax.set_title(
    "Primary mechanistic signature — sample expression"
)

fig.colorbar(
    im,
    ax=ax,
    label="Gene-wise z-score of log2(CPM+1)",
)

fig.tight_layout()

fig.savefig(
    NB10_FIGURES /
    "primary_signature_sample_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

## 25. Stability check: leave-one-out coefficient direction

For the primary logistic model, each LOOCV training fold is fitted and the sign of each coefficient is recorded.

This asks whether gene-direction interpretation is stable to removal of one sample.

In [ ]:
loo = LeaveOneOut()

coefficient_rows = []

X_arr = (
    X_primary
    .to_numpy(
        dtype=float
    )
)

for fold, (
    train_idx,
    test_idx,
) in enumerate(
    loo.split(
        X_arr
    ),
    start=1,
):

    model = clone(
        primary_estimator
    )

    model.fit(
        X_arr[
            train_idx
        ],
        y[
            train_idx
        ],
    )

    if not isinstance(
        model,
        Pipeline,
    ):
        continue

    coef = (
        model
        .named_steps[
            "model"
        ]
        .coef_[0]
    )

    for gene, value in zip(
        primary_genes,
        coef,
    ):

        coefficient_rows.append({
            "fold": fold,
            "held_out_sample": X_primary.index[
                test_idx[0]
            ],
            "gene": gene,
            "coefficient": float(value),
            "positive": bool(
                value > 0
            ),
        })

coef_stability = pd.DataFrame(
    coefficient_rows
)

coef_stability_summary = (
    coef_stability
    .groupby(
        "gene",
        observed=True,
    )
    .agg(
        mean_coefficient=(
            "coefficient",
            "mean",
        ),
        sd_coefficient=(
            "coefficient",
            "std",
        ),
        fraction_positive=(
            "positive",
            "mean",
        ),
    )
    .reset_index()
)

coef_stability_summary[
    "sign_stability"
] = np.maximum(
    coef_stability_summary[
        "fraction_positive"
    ],
    1
    -
    coef_stability_summary[
        "fraction_positive"
    ],
)

coef_stability.to_csv(
    NB10_RESULTS /
    "primary_logistic_LOOCV_coefficients.csv",
    index=False,
)

coef_stability_summary.to_csv(
    NB10_RESULTS /
    "primary_logistic_coefficient_stability.csv",
    index=False,
)

display(
    coef_stability_summary.sort_values(
        "sign_stability",
        ascending=False,
    )
)

## 26. Results summary

In [ ]:
primary_row = (
    primary_metrics
    .iloc[0]
)

results_summary = {
    "n_samples": int(
        len(
            X_primary
        )
    ),
    "n_control": int(
        np.sum(
            y == 0
        )
    ),
    "n_PD": int(
        np.sum(
            y == 1
        )
    ),
    "primary_feature_set": PRIMARY_FEATURE_SET,
    "primary_genes": primary_genes,
    "primary_model": PRIMARY_MODEL,
    "LOOCV_accuracy": float(
        primary_row[
            "accuracy"
        ]
    ),
    "LOOCV_balanced_accuracy": float(
        primary_row[
            "balanced_accuracy"
        ]
    ),
    "LOOCV_precision": float(
        primary_row[
            "precision"
        ]
    ),
    "LOOCV_recall": float(
        primary_row[
            "recall"
        ]
    ),
    "LOOCV_f1": float(
        primary_row[
            "f1"
        ]
    ),
    "LOOCV_MCC": float(
        primary_row[
            "mcc"
        ]
    ),
    "LOOCV_ROC_AUC": float(
        primary_row[
            "roc_auc"
        ]
    ),
    "permutation_p": float(
        permutation_p
    ),
    "approx_bootstrap_auc_95pct": [
        auc_ci_low,
        auc_ci_high,
    ],
}

print(
    json.dumps(
        results_summary,
        indent=2,
    )
)

## 27. Interpretation text for the thesis

The code below prints conservative language that can later be adapted to the Results section.

In [ ]:
print("=== INTERPRETATION ===\n")

print(
    f"The prespecified {PRIMARY_FEATURE_SET} mechanistic signature "
    f"({', '.join(primary_genes)}) was evaluated using leave-one-out "
    f"cross-validation in 10 substantia nigra bulk RNA-seq libraries "
    f"(5 PD and 5 controls)."
)

print(
    f"\nThe primary L2 logistic regression model achieved "
    f"LOOCV accuracy={primary_row['accuracy']:.3f}, "
    f"balanced accuracy={primary_row['balanced_accuracy']:.3f}, "
    f"F1={primary_row['f1']:.3f}, and "
    f"ROC-AUC={primary_row['roc_auc']:.3f}."
)

print(
    f"\nA fixed-feature label-permutation analysis yielded "
    f"p={permutation_p:.4f}."
)

print(
    "\nThese values are exploratory and must not be interpreted as "
    "independent diagnostic performance because the metabolic signature "
    "was partly discovered using the same GSE136666 cohort and the sample "
    "size is extremely small."
)

print(
    "\nIndependent patient-level validation in a larger cohort would be "
    "required before any diagnostic or predictive interpretation."
)

## 28. Export input for Notebook 11 — Explainable AI

Notebook 11 will use:
- fixed primary feature set;
- standardized linear coefficients;
- LOOCV coefficient stability;
- existing biological/network evidence.

Because n=10, SHAP and other XAI outputs must be treated as descriptive.

In [ ]:
xai_feature_table = (
    feature_candidates.loc[
        feature_candidates[
            "signature_gene"
        ].isin(
            primary_genes
        )
    ]
    .copy()
)

if "coef_table" in globals():

    xai_feature_table = (
        xai_feature_table
        .merge(
            coef_table[
                [
                    "gene",
                    "standardized_logistic_coefficient",
                    "abs_coefficient",
                ]
            ],
            left_on="signature_gene",
            right_on="gene",
            how="left",
        )
    )

xai_feature_table = (
    xai_feature_table
    .merge(
        coef_stability_summary,
        left_on="signature_gene",
        right_on="gene",
        how="left",
        suffixes=("", "_stability"),
    )
)

xai_feature_table.to_csv(
    NB10_RESULTS /
    "Notebook11_XAI_feature_input.csv",
    index=False,
)

display(xai_feature_table)

## 29. Manifest

In [ ]:
manifest10 = {
    "dataset": "GSE136666",
    "tissue": "substantia nigra",
    "task": "exploratory PD vs Control classification",
    "n_samples": int(
        X_logcpm.shape[0]
    ),
    "n_control": int(
        np.sum(
            y == 0
        )
    ),
    "n_PD": int(
        np.sum(
            y == 1
        )
    ),
    "feature_source": (
        "Notebook09 mechanistically prioritized "
        "cross-dataset metabolic signature"
    ),
    "normalization": (
        "sample-wise log2(CPM+1) using total raw-library counts"
    ),
    "cross_validation": "leave-one-out cross-validation",
    "scaling": (
        "StandardScaler fitted within each training fold "
        "for linear models"
    ),
    "feature_sets": feature_sets,
    "models": list(
        models.keys()
    ),
    "primary_feature_set": PRIMARY_FEATURE_SET,
    "primary_model": PRIMARY_MODEL,
    "primary_genes": primary_genes,
    "permutations": int(
        N_PERMUTATIONS
    ),
    "primary_metrics": results_summary,
    "independent_validation": False,
    "clinical_classifier": False,
    "critical_limitation": (
        "The signature was partly discovered in GSE136666, "
        "so LOOCV performance on the same cohort is exploratory "
        "and optimistically biased for future clinical prediction."
    ),
    "next_notebook": "11_explainable_AI.ipynb",
}

manifest10_path = (
    NB10_RESULTS /
    "notebook10_manifest.json"
)

with open(
    manifest10_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest10,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    json.dumps(
        manifest10,
        indent=2,
        ensure_ascii=False,
    )
)

## 30. Final checkpoint

In [ ]:
checks10 = {
    "Notebook09 features loaded":
        len(feature_candidates) > 0,

    "GeneID mapping available":
        feature_map["GeneID_clean"].notna().sum() >= 3,

    "10 SN samples identified":
        len(sample_metadata) == 10,

    "5 Control samples":
        int(
            (
                sample_metadata["disease"]
                ==
                "Control"
            ).sum()
        ) == 5,

    "5 PD samples":
        int(
            (
                sample_metadata["disease"]
                ==
                "PD"
            ).sum()
        ) == 5,

    "At least 3 ML genes available":
        X_logcpm.shape[1] >= 3,

    "Multiple feature sets evaluated":
        len(feature_sets) >= 2,

    "All LOOCV predictions generated":
        len(predictions) == (
            len(feature_sets)
            *
            len(models)
            *
            len(sample_metadata)
        ),

    "Primary model evaluated":
        len(primary_metrics) == 1,

    "Permutation test completed":
        len(permuted_aucs) == N_PERMUTATIONS,

    "Coefficient stability generated":
        len(coef_stability_summary) > 0,

    "Notebook11 XAI input exported":
        (
            NB10_RESULTS /
            "Notebook11_XAI_feature_input.csv"
        ).exists(),

    "Manifest exported":
        manifest10_path.exists(),
}

checkpoint10 = (
    pd.Series(
        checks10,
        name="passed",
    )
    .to_frame()
)

checkpoint10[
    "status"
] = (
    checkpoint10[
        "passed"
    ]
    .map({
        True: "PASS",
        False: "FAIL",
    })
)

display(checkpoint10)

checkpoint_path = (
    NB10_RESULTS /
    "notebook10_final_checkpoint.csv"
)

checkpoint10.to_csv(
    checkpoint_path
)

failed = checkpoint10.loc[
    ~checkpoint10[
        "passed"
    ]
]

if len(failed):

    print(
        "\nSome Notebook 10 checks failed:"
    )

    display(failed)

    print(
        "\nNotebook 11 should not be started yet."
    )

else:

    print(
        "\n✓ All Notebook 10 checks passed."
    )

    print(
        "\nNotebook 10 completed successfully."
    )

    print(
        "\nNext:"
    )

    print(
        "11_explainable_AI.ipynb"
    )

## 31. Limitations

1. Only 10 substantia nigra bulk libraries are available.
2. The signature was partly discovered in GSE136666; therefore this is not independent validation.
3. LOOCV reduces sample wastage but does not create new biological replication.
4. Comparing multiple models and feature-set sizes introduces additional optimism if the best result is selectively emphasized.
5. The primary model was specified as Top-5 mechanistic genes + L2 logistic regression to reduce post-hoc model selection.
6. Random Forest results are secondary because nonlinear tree models are particularly unstable at n=10.
7. The permutation test evaluates label association conditional on the fixed signature; it does not correct for all upstream discovery choices.
8. Approximate bootstrap intervals over LOOCV predictions are descriptive, not external-validation confidence intervals.
9. Classification performance must not be described as clinical diagnostic accuracy.
10. Independent validation in a substantially larger patient-level cohort is required.